In [10]:
#!/usr/bin/env python3
"""
ASL Alphabet Classification - 3 Experiment Training Script
===========================================================

This script was built for our ASL project and intentionally reuses ideas from:

1) Class CNN tutorial:
   - simple image-only CNN baseline
   - standard PyTorch train/val/test workflow
   - save-best-model logic
   - train from scratch baseline

2) ENSF 617 assignment training script:
   - clean script structure
   - reusable dataset / dataloader approach
   - CSV + confusion matrix + classification report artifacts
   - transfer learning idea with ResNet-18

IMPORTANT ADAPTATIONS FOR THE ASL DATASET
-----------------------------------------
We did NOT copy those references directly. We changed them to fit the ASL alphabet dataset:

- Changed from MNIST grayscale (1 channel) to ASL RGB images (3 channels)
- Changed from 10 classes (digits) to 29 classes (A-Z, SPACE, DELETE, NOTHING)
- Removed the multimodal text branch from the assignment script
  because ASL classification is an image-only problem here
- Removed RandomVerticalFlip from augmentation because it is unrealistic for hand signs
- Did NOT use filename-derived text because it would be artificial / possibly label leakage
- Replaced stratified split with random_split (dataset is perfectly balanced, 3000 per class)
- Replaced custom ASLImageDataset with ImageFolder + WithTransform wrapper
- Added stronger evaluation outputs for the project report:
    * history.csv
    * classification_report.txt
    * confusion_matrix.npy
    * test_predictions.csv
    * misclassified.csv

PROJECT EXPERIMENTS
-------------------
1) simplecnn
   - Simple CNN baseline adapted from the class tutorial
   - Trained from scratch

2) resnet18_frozen
   - ResNet-18 transfer learning
   - Backbone frozen, only classifier head trained initially
   - Adapted from assignment idea, but image-only

3) resnet18_finetune_aug
   - ResNet-18 with stronger augmentation + staged fine-tuning
   - Better suited for real-world ASL variation

Expected dataset layout
-----------------------
We expect ONE root folder containing 29 class folders:

data_dir/
    A/
    B/
    C/
    ...
    Z/
    SPACE/
    DELETE/
    NOTHING/

Each class folder contains image files.

Example runs
------------
python train_asl_experiments.py --data_dir ./asl_alphabet_train --model simplecnn
python train_asl_experiments.py --data_dir ./asl_alphabet_train --model resnet18_frozen
python train_asl_experiments.py --data_dir ./asl_alphabet_train --model resnet18_finetune_aug
"""

# ============================================================
# Standard library
# ============================================================
import os
import csv
import json
import copy
import random
import argparse
from typing import List, Tuple

# ============================================================
# Numeric / ML
# ============================================================
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder

from sklearn.metrics import classification_report, confusion_matrix, f1_score

from tqdm import tqdm

print(os.listdir('/kaggle/input'))
print(os.listdir('/kaggle/input/datasets'))


['datasets']
['grassknoted']


In [11]:
# ============================================================
# Section 2 - Reproducibility and device helpers
# ============================================================

def set_seed(seed: int = 42):
    """
    Make results more reproducible across runs.
    Helpful when comparing experiments fairly.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device(preferred: str = "cuda") -> torch.device:
    """
    Use GPU if available, otherwise CPU.
    This follows the same general idea as the assignment script.
    """
    if preferred == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


# Set seed and device once here
set_seed(42)
device = get_device("cuda")
print("Using device:", device)

Using device: cuda


In [12]:
# ============================================================
# Section 3 - WithTransform dataset wrapper
# ============================================================

class WithTransform(Dataset):
    """
    Thin wrapper that applies a transform to a Subset returned by random_split.

    WHY THIS IS NEEDED:
    - ImageFolder takes one transform at construction time.
    - But train and eval splits need different transforms.
    - This wrapper lets us attach the correct transform to each split
      after random_split has divided the data.
    """
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        return self.transform(x), y

In [13]:
# ============================================================
# Section 4 - Image transforms for each experiment
# ============================================================

def build_transforms(model_name: str):
    """
    Build transforms based on experiment type.

    IMPORTANT CHANGES FROM REFERENCES:
    - We DO NOT use RandomVerticalFlip from the assignment script
      because vertical flipping is unrealistic for ASL hand signs.
    - We also avoid default horizontal flipping because hand orientation
      may matter in sign data.
    - We use ImageNet normalization so the ResNet transfer learning
      experiments work properly.
    - Resize removed: source images are already 200x200 RGB.
    """
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]

    if model_name == "simplecnn":
        # Based on class CNN tutorial idea, but adapted for ASL
        train_tf = transforms.Compose([
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ])
        eval_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ])

    elif model_name == "resnet18_frozen":
        train_tf = transforms.Compose([
            transforms.RandomRotation(8),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ])
        eval_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ])

    elif model_name == "resnet18_finetune_aug":
        train_tf = transforms.Compose([
            transforms.RandomAffine(
                degrees=12,
                translate=(0.08, 0.08),
                scale=(0.92, 1.08)
            ),
            transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ])
        eval_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ])
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return train_tf, eval_tf

In [14]:
# ============================================================
# Section 5 - Load dataset and inspect
# ============================================================

data_dir = "/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train/asl_alphabet_train"

# ImageFolder replaces the old collect_samples() + ASLImageDataset.
# It expects one subfolder per class, which matches the ASL dataset layout.
# We pass transform=None here because transforms are applied per-split below.
full_dataset = ImageFolder(root=data_dir, transform=None)
class_names = full_dataset.classes
num_classes = len(class_names)

print("Number of classes:", num_classes)
print("Class names:", class_names[:10], "...")
print("Total images:", len(full_dataset))

Number of classes: 29
Class names: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'] ...
Total images: 87000


In [15]:
# ============================================================
# Section 6 - Train/val/test split
# ============================================================

# WHY random_split INSTEAD OF make_stratified_splits:
# The ASL dataset is perfectly balanced (3000 images per class).
# Stratification is only needed when classes are imbalanced.
# random_split is simpler and produces the same class distribution here.

n = len(full_dataset)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

train_subset, val_subset, test_subset = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

print("Train:", len(train_subset))
print("Val:  ", len(val_subset))
print("Test: ", len(test_subset))

Train: 60899
Val:   13050
Test:  13051


In [16]:
# ============================================================
# Section 8 - Simple CNN baseline
# ============================================================

class SimpleASLCNN(nn.Module):
    """
    Simple CNN baseline adapted from the class CNN tutorial.

    CHANGES FROM THE TUTORIAL:
    - Input channels changed from 1 to 3 because ASL images are RGB
    - Output classes changed from 10 to 29
    - Network made slightly deeper because ASL is harder than MNIST
    - Added dropout for regularization
    - Designed for 200x200 input (native ASL image size)
    """

    def __init__(self, num_classes: int = 29):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # CHANGED from 1-channel MNIST
            nn.ReLU(),
            nn.MaxPool2d(2),  # 200 -> 100

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 100 -> 50

            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # extra conv block
            nn.ReLU(),
            nn.MaxPool2d(2),  # 50 -> 25
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 25 * 25, 256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, num_classes),  # CHANGED from 10 to 29
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [17]:
# ============================================================
# Section 9 - ResNet-18 transfer model
# ============================================================

class ResNet18ASL(nn.Module):
    """
    ResNet-18 classifier for ASL.

    BASED ON THE ASSIGNMENT SCRIPT IDEA:
    - keep pretrained ResNet-18 backbone
    - remove DistilBERT
    - remove fusion head
    - replace with image-only classifier for ASL

    This is much more appropriate for this dataset.
    """

    def __init__(self, num_classes: int = 29, freeze_backbone: bool = True):
        super().__init__()

        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = self.backbone.fc.in_features

        # Replace final ImageNet head with ASL classifier head
        self.backbone.fc = nn.Linear(in_features, num_classes)

        if freeze_backbone:
            # Freeze full backbone first
            for param in self.backbone.parameters():
                param.requires_grad = False

            # Re-enable the new FC head for training
            for param in self.backbone.fc.parameters():
                param.requires_grad = True

    def forward(self, x):
        return self.backbone(x)


def unfreeze_resnet_stage4_and_fc(model: ResNet18ASL):
    """
    Used for experiment 3.

    We start frozen, then later unfreeze layer4 + fc.
    This gives a controlled fine-tuning stage.
    """
    for name, param in model.backbone.named_parameters():
        if name.startswith("layer4") or name.startswith("fc"):
            param.requires_grad = True

In [18]:
# ============================================================
# Section 10 - Build selected model
# ============================================================

def build_model(model_name: str, num_classes: int):
    if model_name == "simplecnn":
        return SimpleASLCNN(num_classes=num_classes)

    elif model_name == "resnet18_frozen":
        return ResNet18ASL(num_classes=num_classes, freeze_backbone=True)

    elif model_name == "resnet18_finetune_aug":
        return ResNet18ASL(num_classes=num_classes, freeze_backbone=True)

    else:
        raise ValueError(f"Unknown model name: {model_name}")


def count_trainable_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [19]:
# ============================================================
# Section 12 - Training and evaluation helpers
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Standard training loop.

    Keeps the same general structure used in both the tutorial
    and the assignment:
    - forward
    - compute loss
    - backward
    - optimizer step

    CHANGED: batch is now a (image, label) tuple from WithTransform,
    not a dict. Unpacked directly instead of batch["image"]/batch["label"].
    """
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="train", leave=False):
        images, labels = batch
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()

    avg_loss = total_loss / max(len(loader), 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device, split_name="val"):
    """
    Standard evaluation loop.

    CHANGED: same batch unpacking fix as train_one_epoch.
    """
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc=split_name, leave=False):
        images, labels = batch
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()

    avg_loss = total_loss / max(len(loader), 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


@torch.no_grad()
def predict(model, loader, device):
    """
    Collect predictions from the full test set for later reporting.

    CHANGED: batch is now a (image, label) tuple. Path and filename
    are no longer available from WithTransform, so they are omitted.
    test_predictions.csv and misclassified.csv use index instead.
    """
    model.eval()

    all_preds = []
    all_labels = []

    for batch in tqdm(loader, desc="test", leave=False):
        images, labels = batch
        images = images.to(device)

        logits = model(images)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    return (
        np.array(all_preds),
        np.array(all_labels),
    )

# ============================================================
# Experiment runner helpers
# ============================================================

from pathlib import Path
import pandas as pd

EXPERIMENTS = [
    "simplecnn",
    "resnet18_frozen",
    "resnet18_finetune_aug",
]

BASE_OUTPUT_DIR = Path("./asl_experiment_outputs")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def make_loaders_for_experiment(model_name, train_subset, val_subset, test_subset, batch_size=32):
    train_tf, eval_tf = build_transforms(model_name)

    train_ds = WithTransform(train_subset, train_tf)
    val_ds   = WithTransform(val_subset, eval_tf)
    test_ds  = WithTransform(test_subset, eval_tf)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,  # Set to False for Kaggle environment compatibility
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,  # Set to False for Kaggle environment compatibility
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,  # Set to False for Kaggle environment compatibility
    )

    return train_loader, val_loader, test_loader


def build_optimizer(model_name, model):
    criterion = nn.CrossEntropyLoss()

    if model_name == "simplecnn":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )
    else:
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=2e-4,
            weight_decay=1e-4
        )

    return criterion, optimizer


def write_csv(path: str, rows: List[dict], fieldnames: List[str]):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for row in rows:
            w.writerow(row)


def save_experiment_outputs(
    out_dir,
    model_name,
    best_state,
    history,
    class_names,
    test_loss,
    test_acc,
    macro_f1,
    weighted_f1,
    report,
    cm,
    labels,
    preds,
):
    os.makedirs(out_dir, exist_ok=True)

    # Save model
    torch.save(best_state, os.path.join(out_dir, "best_model.pt"))

    # Save history
    write_csv(
        os.path.join(out_dir, "history.csv"),
        history,
        ["epoch", "train_loss", "train_acc", "val_loss", "val_acc"]
    )

    # Save class names
    with open(os.path.join(out_dir, "class_names.json"), "w", encoding="utf-8") as f:
        json.dump(class_names, f, indent=2)

    # Save classification report
    with open(os.path.join(out_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(report + "\n")

    # Save confusion matrix
    np.save(os.path.join(out_dir, "confusion_matrix.npy"), cm)

    # Save summary metrics
    metrics = {
        "model": model_name,
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "num_classes": len(class_names),
    }

    with open(os.path.join(out_dir, "metrics_summary.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    # Save per-sample predictions
    pred_rows = []
    for idx, (y_true, y_pred) in enumerate(zip(labels.tolist(), preds.tolist())):
        pred_rows.append({
            "index": idx,
            "true": int(y_true),
            "pred": int(y_pred),
            "true_name": class_names[int(y_true)],
            "pred_name": class_names[int(y_pred)],
        })

    write_csv(
        os.path.join(out_dir, "test_predictions.csv"),
        pred_rows,
        ["index", "true", "pred", "true_name", "pred_name"]
    )

    mis_rows = [r for r in pred_rows if r["true"] != r["pred"]]
    write_csv(
        os.path.join(out_dir, "misclassified.csv"),
        mis_rows,
        ["index", "true", "pred", "true_name", "pred_name"]
    )

    return metrics


def run_experiment(model_name, train_subset, val_subset, test_subset, class_names, num_classes, device, epochs=12):
    print("\n" + "=" * 70)
    print(f"Running experiment: {model_name}")
    print("=" * 70)

    # Reproducibility for each experiment
    set_seed(42)

    # Dataloaders
    train_loader, val_loader, test_loader = make_loaders_for_experiment(
        model_name=model_name,
        train_subset=train_subset,
        val_subset=val_subset,
        test_subset=test_subset,
        batch_size=32,
    )

    # Model
    model = build_model(model_name, num_classes=num_classes).to(device)
    print(model)
    print("Trainable parameters:", count_trainable_params(model))

    # Loss / optimizer
    criterion, optimizer = build_optimizer(model_name, model)

    # Training
    best_val_loss = float("inf")
    best_state = None
    history = []

    for epoch in range(epochs):
        if model_name == "resnet18_finetune_aug" and epoch == 3:
            print("\n[Fine-tuning step] Unfreezing ResNet layer4 + fc\n")
            unfreeze_resnet_stage4_and_fc(model)

            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=1e-4,
                weight_decay=1e-4
            )
            print("New trainable parameters:", count_trainable_params(model))

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device, split_name="val")

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

    print("Training finished for:", model_name)

    if best_state is None:
        raise RuntimeError(f"No best model was saved for {model_name}.")

    # Final test evaluation
    model.load_state_dict(best_state)

    test_loss, test_acc = eval_one_epoch(model, test_loader, criterion, device, split_name="test")
    preds, labels = predict(model, test_loader, device)

    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    report = classification_report(labels, preds, target_names=class_names, digits=4)
    cm = confusion_matrix(labels, preds)

    print("Test loss:    ", round(test_loss, 4))
    print("Test accuracy:", round(test_acc, 4))
    print("Macro F1:     ", round(macro_f1, 4))
    print("Weighted F1:  ", round(weighted_f1, 4))

    out_dir = BASE_OUTPUT_DIR / model_name
    metrics = save_experiment_outputs(
        out_dir=str(out_dir),
        model_name=model_name,
        best_state=best_state,
        history=history,
        class_names=class_names,
        test_loss=test_loss,
        test_acc=test_acc,
        macro_f1=macro_f1,
        weighted_f1=weighted_f1,
        report=report,
        cm=cm,
        labels=labels,
        preds=preds,
    )

    print("Saved outputs to:", out_dir)

    return metrics, history

In [20]:
# ============================================================
# Final Section - Run all experiments and build summary table
# ============================================================

all_metrics = []
all_histories = {}

for model_name in EXPERIMENTS:
    metrics, history = run_experiment(
        model_name=model_name,
        train_subset=train_subset,
        val_subset=val_subset,
        test_subset=test_subset,
        class_names=class_names,
        num_classes=num_classes,
        device=device,
        epochs=10,
    )
    all_metrics.append(metrics)
    all_histories[model_name] = history

summary_df = pd.DataFrame(all_metrics)

# Optional: sort best first by test accuracy
summary_df = summary_df.sort_values(by="test_accuracy", ascending=False).reset_index(drop=True)

# Save combined summary
summary_csv_path = BASE_OUTPUT_DIR / "all_experiments_summary.csv"
summary_json_path = BASE_OUTPUT_DIR / "all_experiments_summary.json"

summary_df.to_csv(summary_csv_path, index=False)
summary_df.to_json(summary_json_path, orient="records", indent=2)

print("\nFinal summary table:")
display(summary_df)

print("\nSaved combined summary to:")
print(summary_csv_path)
print(summary_json_path)


Running experiment: simplecnn
SimpleASLCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=80000, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=29, bias=True)
  )
)
Trainable parameters: 20580957


Epoch 01/10 | train_loss=1.2944 | train_acc=0.5823 | val_loss=0.2772 | val_acc=0.9133


Epoch 02/10 | train_loss=0.3839 | train_acc=0.8654 | val_loss=0.1041 | val_acc=0.9663


Epoch 03/10 | train_loss=0.2522 | train_acc=0.9116 | val_loss=0.0792 | val_acc=0.9739


Epoch 04/10 | train_loss=0.1884 | train_acc=0.9361 | val_loss=0.0618 | val_acc=0.9789


...

Epoch 09/10 | train_loss=0.0064 | train_acc=0.9981 | val_loss=0.0012 | val_acc=0.9994


Epoch 10/10 | train_loss=0.0065 | train_acc=0.9976 | val_loss=0.0023 | val_acc=0.9993
Training finished for: resnet18_finetune_aug


Test loss:     0.0004
Test accuracy: 0.9999
Macro F1:      0.9999
Weighted F1:   0.9999
Saved outputs to: asl_experiment_outputs/resnet18_finetune_aug

Final summary table:


,model,test_loss,test_accuracy,macro_f1,weighted_f1,num_classes
0,resnet18_finetune_aug,0.000423,0.999923,0.999927,0.999923,29
1,simplecnn,0.018549,0.993870,0.993938,0.993872,29
2,resnet18_frozen,0.431753,0.883917,0.884253,0.883886,29



Saved combined summary to:
asl_experiment_outputs/all_experiments_summary.csv
asl_experiment_outputs/all_experiments_summary.json


In [21]:
# ============================================================
# Final Metrics Section - Report-friendly table
# ============================================================

report_table = summary_df.copy()

for col in ["test_loss", "test_accuracy", "macro_f1", "weighted_f1"]:
    report_table[col] = report_table[col].map(lambda x: round(float(x), 4))

display(report_table)

,model,test_loss,test_accuracy,macro_f1,weighted_f1,num_classes
0,resnet18_finetune_aug,0.0004,0.9999,0.9999,0.9999,29
1,simplecnn,0.0185,0.9939,0.9939,0.9939,29
2,resnet18_frozen,0.4318,0.8839,0.8843,0.8839,29


In [22]:
import shutil

shutil.make_archive(
    "/kaggle/working/asl_results",
    'zip',
    "/kaggle/working/asl_experiment_outputs"
)

'/kaggle/working/asl_results.zip'